# NayePankh Impact Analytics — EDA and Modeling

This notebook walks through exploratory analysis and predictive modeling for the NayePankh Foundation internship project.

**Run the data pipeline first:**
```bash
python scripts/generate_sample_data.py
python scripts/prepare_data.py
python scripts/train_models.py
```

In [ ]:
from pathlib import Path

import json
import pandas as pd
import plotly.express as px
import plotly.io as pio

ROOT = Path('..').resolve()
pio.renderers.default = 'notebook'

impact_df = pd.read_csv(ROOT / 'data' / 'naye_pankh_sample_impact_data.csv', parse_dates=['date'])
modeling_df = pd.read_csv(ROOT / 'data' / 'modeling_dataset.csv', parse_dates=['date'])
context_df = pd.read_csv(ROOT / 'data' / 'india_state_context.csv')

impact_df.head()

## 1. Dataset Overview

In [ ]:
print(f'Impact records: {len(impact_df):,}')
print(f'Date range: {impact_df.date.min().date()} to {impact_df.date.max().date()}')
print(f'Cities: {sorted(impact_df.city.unique())}')
print(f'Programs: {sorted(impact_df.program.unique())}')
impact_df.describe().T

## 2. Donation and Impact Trends

In [ ]:
monthly = (
    impact_df.groupby(pd.Grouper(key='date', freq='MS'))
    .agg(
        donation_amount=('donation_amount', 'sum'),
        beneficiaries_reached=('beneficiaries_reached', 'sum'),
        volunteer_hours=('volunteer_hours', 'sum'),
    )
    .reset_index()
)

fig = px.line(
    monthly,
    x='date',
    y=['donation_amount', 'beneficiaries_reached'],
    title='Monthly Donations and Beneficiary Reach',
    labels={'value': 'Amount / Count', 'variable': 'Metric'},
)
fig.show()

## 3. Program and City Segmentation

In [ ]:
program_summary = (
    impact_df.groupby('program')
    .agg(
        donation_amount=('donation_amount', 'sum'),
        beneficiaries_reached=('beneficiaries_reached', 'sum'),
        expense_amount=('expense_amount', 'sum'),
    )
    .assign(cost_per_beneficiary=lambda x: x['expense_amount'] / x['beneficiaries_reached'])
    .reset_index()
)

px.bar(
    program_summary.sort_values('cost_per_beneficiary'),
    x='program',
    y='cost_per_beneficiary',
    color='program',
    title='Cost per Beneficiary by Program',
).show()

city_summary = impact_df.groupby('city')['beneficiaries_reached'].sum().reset_index()
px.bar(city_summary, x='city', y='beneficiaries_reached', title='Total Beneficiaries by City').show()

## 4. Seasonality and Donor Mix

In [ ]:
seasonal = impact_df.groupby('is_festive_season')['donation_amount'].mean().reset_index()
seasonal['season'] = seasonal['is_festive_season'].map({0: 'Non-Festive', 1: 'Festive (Aug-Dec)'})
px.bar(seasonal, x='season', y='donation_amount', title='Average Donation: Festive vs Non-Festive').show()

donor_mix = impact_df.groupby('donor_type')['donation_amount'].sum().reset_index()
px.pie(donor_mix, names='donor_type', values='donation_amount', title='Donation Share by Donor Type', hole=0.45).show()

## 5. Geographic Context

In [ ]:
city_context = (
    impact_df.groupby(['city', 'state'], as_index=False)
    .agg(beneficiaries_reached=('beneficiaries_reached', 'sum'), volunteer_hours=('volunteer_hours', 'sum'))
    .merge(context_df, on='state', how='left')
)
city_context['volunteers_per_1000'] = city_context['volunteer_hours'] / city_context['beneficiaries_reached'] * 1000

px.scatter(
    city_context,
    x='multidimensional_poverty_index',
    y='volunteers_per_1000',
    size='beneficiaries_reached',
    color='city',
    title='Regional Need vs Volunteer Support',
).show()

## 6. Model Comparison Results

In [ ]:
metrics_path = ROOT / 'outputs' / 'model_metrics.json'
if metrics_path.exists():
    with metrics_path.open() as handle:
        metrics_payload = json.load(handle)
    results_df = pd.DataFrame(metrics_payload['results'])
    display(results_df)
    print('\nRecommended models:')
    for task, model in metrics_payload['recommended_models'].items():
        print(f'- {task}: {model}')
else:
    print('Run python scripts/train_models.py to generate model metrics.')

## 7. Forecast vs Actual

In [ ]:
forecast_path = ROOT / 'outputs' / 'forecast_vs_actual.csv'
if forecast_path.exists():
    forecast_df = pd.read_csv(forecast_path, parse_dates=['date'])
    donation_forecast = forecast_df[forecast_df['target'] == 'donation_amount']
    monthly_forecast = donation_forecast.groupby('date', as_index=False).sum(numeric_only=True)
    px.line(
        monthly_forecast,
        x='date',
        y=['actual', 'predicted'],
        title='2025 Donation Forecast vs Actual',
        labels={'value': 'Donation Amount', 'variable': 'Series'},
    ).show()

## 8. Key Takeaways

1. Festive months drive higher average donations and should guide campaign planning.
2. Program efficiency varies materially — menstrual hygiene and food distribution are strong efficiency candidates.
3. Kanpur and Delhi lead in reach; some high-need states may still be under-resourced on volunteers.
4. XGBoost outperforms linear baselines for forecasting donations and beneficiary reach.
5. The scenario simulator in the Streamlit dashboard turns these models into planning tools for field teams.